# 🛡️ Citadel PII Processing - Testing Center

## Test PII Anonymization, Deanonymization, and Blocking capabilities!

Use this Jupyter notebook to verify PII processing in Citadel Access Contracts, including:
- **Use Case 1**: PII Anonymization/Deanonymization with state saving
- **Use Case 2**: PII Blocking with detection reporting
- Testing with various PII types (names, emails, phone numbers, credit cards, IBANs, etc.)
- Validating custom regex pattern detection
- Verifying Event Hub logging for PII state saving

> **Note:** This notebook assumes you have already deployed your Citadel Governance Hub with PII processing capabilities enabled. 
> Ensure Azure AI Language Service is configured and the PII policy fragments are deployed.

## Azure Prerequisites

- An Azure AI Language Service instance with PII detection enabled
- Event Hub configured for PII state logging (if testing state saving)
- APIM policy fragments deployed: `pii-anonymization`, `pii-deanonymization`, `pii-state-saving`
- Managed Identity configured with access to Language Service

### Shared workflow and safety

Run from the repository root or `validation`. Explicit `overrides` win over the
selected azd environment. Annotate deployed model operations with `capabilities`
(or `VALIDATION_MODEL_CAPABILITIES`); unknown capabilities block required coverage.
Image operations use `image-generation` and `image-edit`; affinity uses `responses`.
Definitions and policy XML remain visible. Preview the generated **`.bicepparam`**
in one cell, optionally edit it, then deploy it in the following cell; the shared
deploy helper compiles the actual parameter file before submission.
Generated artifacts live under ignored `validation/.runs`. Tokens/keys stay in
memory. `run.finish()` raises for required FAIL/BLOCKED/empty runs. Cleanup is off
and can only delete resources absent before this run. Cloud execution has real cost
and side effects; local source validation is not a cloud feature verdict.

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group`, `location`, and the optional Cosmos account from your active `azd` environment. Works when the accelerator was deployed with `azd up`.
- `False` — set explicit resource coordinates in `overrides`.

### 0️⃣ Initialize Notebook Variables

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.

Initialization requires a Cosmos endpoint/database/container (and an environment key for key authentication), or captured `pii_state_evidence`. Correlated masking/restoration evidence is still checked after requests; supplying an evidence list alone is not a passing result.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Additional scenario inputs

- `cosmos_account_key` reads `VALIDATION_COSMOS_KEY`. Supply it before Run All, for example `os.environ['VALIDATION_COSMOS_KEY'] = secret_store_value`; use an approved secret store and never save secret values in notebook source.

### Scenario-specific input examples


The Cosmos account endpoint is loaded from azd when available. For a different
evidence source, use for example
`cosmos_account_endpoint = "https://my-cosmos.documents.azure.com:443/"`.
PII state evidence must match this run's correlation IDs and contain real
observations; do not fabricate evidence to make checks pass. `pii_state_evidence`
accepts retrieved records, for example
`pii_state_evidence = [record_from_your_telemetry_query]`.
`enable_cosmos_analytics = True` enables the optional analysis scenario.
`grant_cosmos_data_access = True` explicitly permits the role-assignment task;
leave it `False` if your identity already has the required read access.


In [1]:
from pathlib import Path
import sys
repo_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(repo_root / "shared"))
import utils
with utils.notebook_step('0️⃣ Initialize Notebook Variables', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    import os, sys, json, time, uuid, base64, requests
    from pathlib import Path
    from apimtools import APIMClientTool
    from validationtools import ValidationContext, ValidationRun, RequestTransport, ValidationBlocked
    from contracttools import write_parameters, deploy_contract, access_policy, preview
    from scenariotools import access_parameters, contract_identity, cleanup_products, policy_xml, completion_text, save_manifest

    init_from_azd = True
    environment = None  # Explicit azd environment, or the active environment.
    overrides = {}  # Explicit values win: resource_group/location/subscription_id/apim_name.
    # Supply capability metadata, not name patterns, e.g.
    # overrides["capabilities"] = {"your-deployed-model": ["chat", "responses"]}
    run_id = uuid.uuid4().hex[:12]
    cleanup_delete_contract = False

    model_override = None
    inference_api_version = "2024-05-01-preview"
    openai_api_version = "2024-12-01-preview"
    targetInferenceApi = "models"
    cosmos_account_endpoint = ""
    cosmos_database_name = None  # azd value, otherwise the deployed template's database.
    cosmos_container_name = None  # azd value, otherwise the deployed PII container.
    cosmos_auth_mode = "aad"
    cosmos_account_key = os.environ.get("VALIDATION_COSMOS_KEY", "")
    enable_cosmos_analytics = True  # Charts and LLM telemetry reasoning are optional.
    grant_cosmos_data_access = True  # Explicit permission-changing opt-in.
    pii_state_evidence = []  # Alternative: real captured PII_Processing events for this run.
    evidence_poll_attempts = 6
    evidence_poll_seconds = 10
    top_n_records = 100
    filter_product_id = None
    filter_subscription_id = None

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

### 1️⃣ Verify Azure CLI and Connected Subscription

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [2]:
with utils.notebook_step('1️⃣ Verify Azure CLI and Connected Subscription', inputs={'cosmos_account_endpoint': globals().get('cosmos_account_endpoint'), 'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'targetInferenceApi': globals().get('targetInferenceApi'), 'cosmos_auth_mode': globals().get('cosmos_auth_mode'), 'run_id': globals().get('run_id'), 'overrides': globals().get('overrides')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    run = ValidationRun("pii-processing")
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=init_from_azd, operation_overrides=operation_overrides, skipped_models=skipped_models)
    cosmos_account_key = cosmos_account_key or context.config.get("VALIDATION_COSMOS_KEY", "")
    mandatory_inputs = {
        "gateway.subscription_id": context.config.get("subscription_id"),
        "gateway.resource_group": context.config.get("resource_group"),
        "gateway.location": context.config.get("location"),
        "gateway.apim_name": context.config.get("apim_name"),
    }
    cosmos_account_endpoint = cosmos_account_endpoint or (context.config.get("cosmos_account_endpoint") or context.config.get("COSMOS_DB_ACCOUNT_ENDPOINT") or context.config.get("COSMOS_ACCOUNT_ENDPOINT", ""))
    cosmos_database_name = cosmos_database_name or context.config.get("cosmos_database_name") or "ai-usage-db"
    cosmos_container_name = cosmos_container_name or context.config.get("cosmos_pii_container_name") or "pii-usage-container"
    cosmos_account_name = context.config.get("cosmos_account_name") or context.config.get("cosmos_db_account_name")
    if not cosmos_account_endpoint and cosmos_account_name:
        cosmos_account_endpoint = f"https://{cosmos_account_name}.documents.azure.com"
    if not pii_state_evidence or enable_cosmos_analytics:
        mandatory_inputs.update({
            "pii.cosmos_account_endpoint or pii_state_evidence": cosmos_account_endpoint,
            "pii.cosmos_database_name": cosmos_database_name,
            "pii.cosmos_container_name": cosmos_container_name,
            "pii.cosmos_auth_mode": cosmos_auth_mode,
        })
        if cosmos_auth_mode == "key":
            mandatory_inputs["pii.cosmos_account_key"] = cosmos_account_key
    if targetInferenceApi not in ("models", "openai"):
        raise ValueError("targetInferenceApi must be models or openai.")
    if cosmos_auth_mode not in ("aad", "key"):
        raise ValueError("cosmos_auth_mode must be aad or key.")
    utils.require_inputs(mandatory_inputs)
    utils.require_capabilities(context.catalog, context.catalog.models_for("chat"))
    governance_hub_resource_group = context.config["resource_group"]
    location = context.config["location"]
    subscription_id = context.config["subscription_id"]
    llm_backends_config = context.config["llm_backends"]
    run_dir = repo_root / "validation" / ".runs" / "pii-processing" / run_id
    transport = RequestTransport(timeout=120)
    try:
        model_name = context.catalog.select("chat", override=model_override)
    except ValidationBlocked as exc:
        run.blocked("model-preflight", str(exc))
        raise

    cosmos_account_endpoint = cosmos_account_endpoint or (context.config.get("cosmos_account_endpoint") or context.config.get("COSMOS_DB_ACCOUNT_ENDPOINT") or context.config.get("COSMOS_ACCOUNT_ENDPOINT", ""))
    cosmos_database_name = cosmos_database_name or context.config.get("cosmos_database_name") or "ai-usage-db"
    cosmos_container_name = cosmos_container_name or context.config.get("cosmos_pii_container_name") or "pii-usage-container"
    cosmos_account_name = context.config.get("cosmos_account_name") or context.config.get("cosmos_db_account_name")
    if not cosmos_account_endpoint and cosmos_account_name:
        cosmos_account_endpoint = f"https://{cosmos_account_name}.documents.azure.com"

    utils.format_output('1️⃣ Verify Azure CLI and Connected Subscription', {'value_1': {"environment": environment or "active", "resource_group": governance_hub_resource_group,
           "location": location, "subscription_id": subscription_id}}, status='info', part='RESULT')
    utils.format_output('1️⃣ Verify Azure CLI and Connected Subscription', {'value_1': context.catalog.report()}, status='info', part='RESULT')

    run_dir.mkdir(parents=True, exist_ok=True)


<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected with PII policy fragments deployed.

### 2️⃣ Initialize APIM Client Tool

Prepare the 2️⃣ Initialize APIM Client Tool state shown below for its subsequent feature-specific requests and assertions.

**Input:** `governance_hub_resource_group`, `targetInferenceApi`, `openai_api_version`, `inference_api_version`, `model_name`

**Output:** `apimClientTool`, `gateway_url`, `azure_endpoint`, `apim_resource_gateway_url`, `apim_name`, `preexisting_products` are available to the following cells.


In [3]:
with utils.notebook_step('2️⃣ Initialize APIM Client Tool', inputs={'governance_hub_resource_group': globals().get('governance_hub_resource_group'), 'targetInferenceApi': globals().get('targetInferenceApi'), 'openai_api_version': globals().get('openai_api_version'), 'inference_api_version': globals().get('inference_api_version'), 'model_name': globals().get('model_name')}, expected='`apimClientTool`, `gateway_url`, `azure_endpoint`, `apim_resource_gateway_url`, `apim_name`, `preexisting_products` are available to the following cells.', namespace=globals()):
    apimClientTool = APIMClientTool(governance_hub_resource_group, apim_resource_name=context.config["apim_name"])
    apimClientTool.initialize_context(context.account, enumerate_subscriptions=False)

    gateway_url = str(apimClientTool.apim_resource_gateway_url).rstrip("/")
    azure_endpoint = gateway_url + "/"
    apim_resource_gateway_url = gateway_url
    apim_name = apimClientTool.apim_resource_name
    preexisting_products = {p.name for p in apimClientTool.client.product.list_by_service(
        governance_hub_resource_group, apim_name)}
    owned_products = set()

    from urllib.parse import quote
    if targetInferenceApi not in ("models", "openai"):
        raise ValueError("targetInferenceApi must be models or openai.")
    pii_api = apimClientTool.discover_api_exact(targetInferenceApi)
    if targetInferenceApi == "openai":
        chat_completions_url = f"{pii_api['url'].rstrip('/')}/deployments/{quote(model_name, safe='')}/chat/completions?api-version={openai_api_version}"
    else:
        chat_completions_url = f"{pii_api['url'].rstrip('/')}/chat/completions?api-version={inference_api_version}"


---
## 🔐 Use Case 1: PII Anonymization & Deanonymization

This use case tests PII masking where:
- Inbound requests have PII replaced with placeholders (e.g., `<Person_0>`, `<Email_0>`)
- Outbound responses have placeholders restored with original PII values
- State saving logs all PII processing to Event Hub for auditing
---

<a id='3.1'></a>
### 3️⃣.1 Define PII Masking Access Contract

Create an access contract with PII anonymization/deanonymization enabled and state saving turned on.

### 3️⃣.1 Define PII Masking Access Contract

Prepare the 3️⃣.1 Define PII Masking Access Contract state shown below for its subsequent feature-specific requests and assertions.

**Input:** `run_id`

**Output:** `timestamp`, `pii_masking_contract` are available to the following cells.


In [4]:
with utils.notebook_step('3️⃣.1 Define PII Masking Access Contract', inputs={'run_id': globals().get('run_id')}, expected='`timestamp`, `pii_masking_contract` are available to the following cells.', namespace=globals()):
    timestamp = run_id

    # PII Masking Use Case Configuration
    pii_masking_contract = {
        "name": f"pii-masking-contract-{timestamp}",
        "business_unit": "HR",
        "use_case_name": "PIIMasking-" + run_id,
        "environment": "DEV",
        "use_keyvault": False,
        "endpoint_secret": "HR-PII-LLM-ENDPOINT",
        "apikey_secret": "HR-PII-LLM-KEY",
        "description": "HR PII Masking - Anonymization/Deanonymization with State Saving",
        "pii_config": {
            "mode": "anonymization",
            "confidence_threshold": "0.8",
            "entity_exclusions": "PersonType",
            "detection_language": "en",
            "state_saving_enabled": True
        }
    }

    utils.print_info(f"PII Masking Contract Configuration:")
    utils.print_info(f"  Business Unit: {pii_masking_contract['business_unit']}")
    utils.print_info(f"  Use Case: {pii_masking_contract['use_case_name']}")
    utils.print_info(f"  PII Mode: {pii_masking_contract['pii_config']['mode']}")
    utils.print_info(f"  State Saving: {pii_masking_contract['pii_config']['state_saving_enabled']}")
    utils.print_info(f"  Product ID: LLM-{pii_masking_contract['business_unit']}-{pii_masking_contract['use_case_name']}-{pii_masking_contract['environment']}")


<a id='3.2'></a>
### 3️⃣.2 Create PII Masking Product Policy

Generate a custom product policy XML that enables PII anonymization/deanonymization with custom regex patterns and state saving.

### 3️⃣.2 Create PII Masking Product Policy

Construct or inspect the 3️⃣.2 Create PII Masking Product Policy policy, retaining the feature-specific grants and controls shown below.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** The policy XML/configuration and its visible inspection results; no deployment is hidden in this step.


In [5]:
with utils.notebook_step('3️⃣.2 Create PII Masking Product Policy', inputs={}, expected='The policy XML/configuration and its visible inspection results; no deployment is hidden in this step.', namespace=globals()):
    pii_masking_policy = '''<policies>
        <inbound>
            <base />
            <!-- Enable PII Anonymization -->
            <set-variable name="piiAnonymizationEnabled" value="true" />

            <choose>
                <when condition="@(context.Variables.GetValueOrDefault<string>(\"piiAnonymizationEnabled\") == \"true\")">

                    <!-- Configure PII detection settings -->
                    <set-variable name="piiConfidenceThreshold" value="0.8" />
                    <set-variable name="piiEntityCategoryExclusions" value="PersonType" />
                    <set-variable name="piiDetectionLanguage" value="en" />

                    <!-- Configure custom regex patterns for additional PII detection -->
                    <set-variable name="piiRegexPatterns" value="@{
                        var patterns = new JArray {
                            new JObject {
                                [\"pattern\"] = @\"\\b\\d{4}[- ]?\\d{4}[- ]?\\d{4}[- ]?\\d{4}\\b\",
                                [\"category\"] = \"CREDIT_CARD\"
                            },
                            new JObject {
                                [\"pattern\"] = @\"\\b[A-Z]{2}\\d{6}[A-Z]\\b\",
                                [\"category\"] = \"PASSPORT_NUMBER\"
                            },
                            new JObject {
                                [\"pattern\"] = @\"\\b784-\\d{4}-\\d{7}-\\d{1}\\b\",
                                [\"category\"] = \"EMIRATES_ID\"
                            }
                        };
                        return patterns.ToString();
                    }" />

                    <!-- Capture request body for PII processing -->
                    <set-variable name="piiInputContent" value="@(context.Request.Body.As<string>(preserveContent: true))" />

                    <!-- Apply PII anonymization -->
                    <include-fragment fragment-id="pii-anonymization" />

                    <!-- Replace request body with anonymized content -->
                    <set-body>@(context.Variables.GetValueOrDefault<string>("piiAnonymizedContent"))</set-body>
                </when>
            </choose>
        </inbound>
        <backend>
            <base />
        </backend>
        <outbound>
            <base />
            <!-- Store response body before processing -->
            <set-variable name="responseBodyContent" value="@(context.Response.Body.As<string>(preserveContent: true))" />

            <choose>
                <when condition="@(context.Variables.GetValueOrDefault<string>(\"piiAnonymizationEnabled\") == \"true\" && 
                                context.Variables.ContainsKey(\"piiMappings\"))">

                    <!-- Set input for deanonymization -->
                    <set-variable name="piiDeanonymizeContentInput" value="@(context.Variables.GetValueOrDefault<string>(\"responseBodyContent\"))" />

                    <!-- Apply PII deanonymization -->
                    <include-fragment fragment-id="pii-deanonymization" />

                    <!-- Enable PII processing audit logging to Event Hub -->
                    <set-variable name="piiStateSavingEnabled" value="true" />
                    <set-variable name="originalRequest" value="@(context.Variables.GetValueOrDefault<string>(\"piiInputContent\"))" />
                    <set-variable name="originalResponse" value="@(context.Variables.GetValueOrDefault<string>(\"responseBodyContent\"))" />
                    <include-fragment fragment-id="pii-state-saving" />

                    <!-- Replace response with deanonymized content -->
                    <set-body>@(context.Variables.GetValueOrDefault<string>("piiDeanonymizedContentOutput"))</set-body>
                </when>
                <otherwise>
                    <!-- Pass through original response -->
                    <set-body>@(context.Variables.GetValueOrDefault<string>("responseBodyContent"))</set-body>
                </otherwise>
            </choose>
        </outbound>
        <on-error>
            <base />
        </on-error>
    </policies>'''
    pii_masking_policy = policy_xml(pii_masking_policy)
    utils.format_output('3️⃣.2 Create PII Masking Product Policy', {'pii_masking_policy': pii_masking_policy}, status='info', part='RESULT')


<a id='3.3'></a>
### 3️⃣.3 Create Parameter File and Deploy PII Masking Contract

### 3️⃣.3 Create Parameter File and Deploy PII Masking Contract

Generate the editable parameter artifact for 3️⃣.3 Create Parameter File and Deploy PII Masking Contract and display the proposed configuration without deploying it.

**Input:** `pii_masking_contract`, `pii_masking_policy`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [6]:
with utils.notebook_step('3️⃣.3 Create Parameter File and Deploy PII Masking Contract', inputs={'pii_masking_contract': globals().get('pii_masking_contract'), 'pii_masking_policy': globals().get('pii_masking_policy'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    masking_parameters = access_parameters(context, apimClientTool, pii_masking_contract, pii_masking_policy,
        key_vault=None)

    masking_params_file = write_parameters(
        repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
        masking_parameters, run_dir / "masking")
    utils.format_output('3️⃣.3 Create Parameter File and Deploy PII Masking Contract', {'value_1': preview(masking_params_file)}, status='info', part='RESULT')


<a id='3.4'></a>
### 3️⃣.4 Retrieve API Key for PII Masking Contract

### 3️⃣.4 Retrieve API Key for PII Masking Contract

Deploy the reviewed artifact for 3️⃣.4 Retrieve API Key for PII Masking Contract, then inspect its real resource outputs and ownership.

**Input:** `masking_params_file`, `location`, `preexisting_products`, `owned_products`, `pii_masking_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [7]:
with utils.notebook_step('3️⃣.4 Retrieve API Key for PII Masking Contract', inputs={'masking_params_file': globals().get('masking_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'pii_masking_contract': globals().get('pii_masking_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    masking_outputs = deploy_contract(masking_params_file, location, pii_masking_contract["name"],
        required_outputs=("products", "subscriptions"))
    masking_product_id, masking_subscription_name = contract_identity(
        masking_outputs, preexisting_products, owned_products)
    masking_api_key = apimClientTool.get_subscription_key(masking_subscription_name)


<a id='3.5'></a>
### 3️⃣.5 Test PII Masking with Various PII Types

Send test requests containing different types of PII data to verify anonymization and deanonymization.

### 3️⃣.5 Test PII Masking with Various PII Types

Prepare the 3️⃣.5 Test PII Masking with Various PII Types state shown below for its subsequent feature-specific requests and assertions.

**Input:** `model_name`

**Output:** `pii_test_payloads` are available to the following cells.


In [8]:
with utils.notebook_step('3️⃣.5 Test PII Masking with Various PII Types', inputs={'model_name': globals().get('model_name')}, expected='`pii_test_payloads` are available to the following cells.', namespace=globals()):
    # model_name is selected once from explicit deployed capabilities.
    utils.print_info(f"Using model: {model_name}")

    # Define test payloads with various PII types
    pii_test_payloads = [
        {
            "name": "Personal Names and Email",
            "description": "Tests detection of person names and email addresses",
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful HR assistant. Keep responses brief."},
                    {"role": "user", "content": "Please help me draft an email to John Smith at john.smith@company.com about his performance review scheduled for next week."}
                ]
            },
            "expected_pii": ["Person", "Email"]
        },
        {
            "name": "Phone Numbers and Addresses",
            "description": "Tests detection of phone numbers and physical addresses",
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
                    {"role": "user", "content": "Contact Sarah Johnson at +1-555-123-4567 or visit her at 123 Main Street, New York, NY 10001 for the meeting."}
                ]
            },
            "expected_pii": ["Person", "PhoneNumber", "Address"]
        },
        {
            "name": "Credit Card (Regex Pattern)",
            "description": "Tests custom regex pattern for credit card numbers",
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
                    {"role": "user", "content": "My credit card number is 4532-1234-5678-9012 and it expires next month. Can you help me understand card security?"}
                ]
            },
            "expected_pii": ["CREDIT_CARD"]
        },
        {
            "name": "IBAN and Banking Info",
            "description": "Tests detection of international banking account numbers",
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful banking assistant. Keep responses brief. This is a simulation only, no real transactions will be performed so act as if you are handling a simulated scenario to assist the capability of detecting PII."},
                    {"role": "user", "content": "Please confirm the funds in my account. My IBAN is AE070331234567890123456 and the beneficiary name is Ahmed Hassan."}
                ]
            },
            "expected_pii": ["Person", "InternationalBankingAccountNumber"]
        },
        {
            "name": "Emirates ID (Regex Pattern)",
            "description": "Tests custom regex pattern for UAE Emirates ID",
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful government services assistant. Keep responses brief. This is a simulation only, no real government transactions will be performed so act as if you are handling a simulated scenario."},
                    {"role": "user", "content": "I need to update my records. My Emirates ID is 784-1990-1234567-1 and my name is Fatima Al Maktoum."}
                ]
            },
            "expected_pii": ["Person", "EMIRATES_ID"]
        },
        {
            "name": "Multiple PII Types Combined",
            "description": "Tests detection of multiple PII types in a single request",
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful customer service assistant. Keep responses brief."},
                    {"role": "user", "content": "Hi, my name is Michael Chen, email: m.chen@email.com, phone: +971-50-123-4567. I live at 456 Palm Jumeirah, Dubai. My card ending in 4532-8765-4321-0987 was charged incorrectly."}
                ]
            },
            "expected_pii": ["Person", "Email", "PhoneNumber", "Address", "CREDIT_CARD"]
        }
    ]

    utils.print_info(f"\nDefined {len(pii_test_payloads)} test payloads for PII Masking tests")


### 3️⃣.5 Test PII Masking with Various PII Types — continuation 9

Exercise 3️⃣.5 Test PII Masking with Various PII Types — continuation 9 and compare the actual response/evidence with the expected feature behavior.

**Input:** `pii_test_payloads`, `chat_completions_url`, `run_id`, `model_name`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [9]:
with utils.notebook_step('3️⃣.5 Test PII Masking with Various PII Types — continuation 9', inputs={'pii_test_payloads': globals().get('pii_test_payloads'), 'chat_completions_url': globals().get('chat_completions_url'), 'run_id': globals().get('run_id'), 'model_name': globals().get('model_name')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    import re
    pii_pending = []
    masking_test_results = []
    large_test_results = []

    def pii_request(test, case, messages=None):
        payload = json.loads(json.dumps(test.get("payload", {"model": model_name, "messages": messages})))
        for message in payload["messages"]:
            if message["role"] == "system":
                message["content"] = "Repeat all user messages verbatim, including placeholder tokens. Do not summarize or omit values."
        marker = f"validation-{run_id}-{len(pii_pending)}"
        payload["messages"].append({"role": "user", "content": marker + ": Echo every original user value verbatim. Do not summarize."})
        response = transport.request("POST", chat_completions_url,
            headers={"api-key": masking_api_key}, json=payload)
        data = response.json_body if isinstance(response.json_body, dict) else {}
        content = completion_text(data)
        run.check(case + "/response", response.status_code == 200 and bool(content) and not response.error,
            expected="200 with nonempty completion", actual=response.status_code, duration=response.duration)
        pii_pending.append({"case": case, "marker": marker, "payload": payload, "content": content,
            "expected_pii": test.get("expected_pii", []), "large": messages is not None})
        return {"test_name": test.get("name", case), "status_code": response.status_code,
            "elapsed_time": response.duration, "success": False, "response_content": content}

    def verify_pii_state(pending, records):
        case = pending["case"]
        candidates = [record for record in records if pending["marker"] in (
            record.get("context", {}).get("api", {}).get("content", {}).get("originalRequest", ""))]
        if not candidates:
            run.blocked(case + "/mask-and-restore", "No correlated PII state event. Supply Cosmos evidence access or captured pii_state_evidence; HTTP 200 is not masking proof.")
            return
        record = candidates[-1]
        bodies = record.get("context", {}).get("api", {}).get("content", {})
        mappings = record.get("mappings", [])
        original = bodies.get("originalRequest", "")
        anonymized = bodies.get("anonymizedRequest", "")
        before = bodies.get("originalResponse", "")
        restored = bodies.get("deanonymizedResponse", "")
        replaced = [m for m in mappings if m.get("original") and m.get("placeholder") in anonymized
                    and m["original"] not in anonymized]
        restoration = [m for m in mappings if m.get("placeholder") in before and m.get("original") in restored
                       and m["placeholder"] not in restored]
        categories = record.get("entityCategories", {})
        expected = pending["expected_pii"]
        actual_categories = set(categories)
        expected_found = all(any(e.lower() in c.lower() or c.lower() in e.lower()
                                for c in actual_categories) for e in expected)
        try:
            structure_ok = json.loads(original) == pending["payload"] and isinstance(json.loads(anonymized), dict)
            restored_content = json.loads(restored).get("choices", [{}])[0].get("message", {}).get("content", "")
        except (ValueError, TypeError):
            structure_ok, restored_content = False, ""
        output_ok = restored_content == pending["content"] and not re.search(r"<[A-Za-z0-9]+_\d+>", pending["content"])
        run.check(case + "/mask-and-restore",
            bool(replaced) and bool(restoration) and expected_found and structure_ok and output_ok,
            expected="Correlated original/anonymized request and placeholder/original response restoration",
            actual={"masked": len(replaced), "restored": len(restoration), "categories": list(categories)},
            evidence_id=record.get("id"), model=model_name)
        if pending["large"]:
            serialized = json.dumps(pending["payload"])
            # Prove entities on both sides of the actual service document boundary were masked.
            positions = [original.find(m["original"]) for m in replaced]
            boundary_proof = any(0 <= p < 5120 for p in positions) and any(p >= 5120 for p in positions)
            run.check(case + "/large-boundaries", len(original) > 5120 and boundary_proof and structure_ok,
                expected="Valid full JSON >5120 chars; masking observed before and after service boundary",
                actual={"actual_chars": len(original), "mapped_positions": positions},
                measured_request_chars=len(serialized))

    for test in pii_test_payloads:
        utils.format_output('3️⃣.5 Test PII Masking with Various PII Types — continuation 9', {'value_1': test["name"], 'value_2': test["description"], 'value_3': test["expected_pii"]}, status='info', part='RESULT')
        masking_test_results.append(pii_request(test, "masking/" + test["name"]))


---
## 🧪 Use Case 3: Large Request / Chunked PII Processing

Azure AI Language Service enforces a hard **5,120-character-per-document** limit for PII detection. This use case validates that the `pii-anonymization` fragment transparently handles payloads larger than that limit by:

- **Chunking** the request body into overlapping chunks (≤ `piiMaxChunkSize`, default 5,000; overlap `piiChunkOverlap`, default 250)
- **Batching** up to **5 chunks per Language Service request** and spanning multiple requests (up to ~125,000 chars)
- **Aggregating** detected entities across all chunks and applying a **global text replacement on the full original body** — so request structure and reconstruction fidelity are preserved

The tests below load realistic, PII-rich conversation histories from `validation/pii-large-samples/`, grow them to two target sizes (one in the **10K–25K** band, one **35K+**), send them through the PII Masking access contract, and capture the **latency** of each call while verifying the response is fully deanonymized (no `<Category_n>` placeholders leak back to the client).

> Set `enableLargeRequestTesting = True` in the configuration cell below to run these tests. This required feature coverage is enabled by default. HTTP success and estimated chunk counts are not proof; the final summary requires correlated state evidence with masking before and after the document boundary.
---


### 🧪 Use Case 3: Large Request / Chunked PII Processing

Prepare the 🧪 Use Case 3: Large Request / Chunked PII Processing state shown below for its subsequent feature-specific requests and assertions.

**Input:** `model_name`

**Output:** `enableLargeRequestTesting`, `LANGUAGE_SERVICE_DOC_LIMIT`, `PII_MAX_CHUNK_SIZE`, `PII_CHUNK_OVERLAP`, `large_test_targets`, `t` are available to the following cells.


In [10]:
with utils.notebook_step('🧪 Use Case 3: Large Request / Chunked PII Processing', inputs={'model_name': globals().get('model_name')}, expected='`enableLargeRequestTesting`, `LANGUAGE_SERVICE_DOC_LIMIT`, `PII_MAX_CHUNK_SIZE`, `PII_CHUNK_OVERLAP`, `large_test_targets`, `t` are available to the following cells.', namespace=globals()):
    import re

    # ============================================================================
    # 🧪 Use Case 3: Large Request / Chunked PII Processing
    # ----------------------------------------------------------------------------
    # Azure AI Language Service caps PII detection at 5,120 characters PER DOCUMENT.
    # The pii-anonymization fragment now splits larger payloads into overlapping
    # chunks, batches up to 5 chunks per Language Service request, and analyzes
    # across multiple requests (up to ~125,000 chars). These tests exercise that
    # path with realistic, PII-rich conversation histories loaded from disk.
    #
    # Toggle the flag below to run the large-document tests.
    # ============================================================================
    enableLargeRequestTesting = True   # Set True to run the large-document chunking tests

    LANGUAGE_SERVICE_DOC_LIMIT = 5120   # chars per document (service hard limit)
    PII_MAX_CHUNK_SIZE = 5000           # matches the fragment default (piiMaxChunkSize)
    PII_CHUNK_OVERLAP = 250             # matches the fragment default (piiChunkOverlap)

    # Realistic seed conversations placed under validation/pii-large-samples/.
    # Each is grown to a target size to simulate a long chat history while keeping
    # the JSON body structurally valid and full of varied PII (names, emails,
    # phones, IBANs, cards, national IDs, addresses, passports, SSNs, IPs, ...).
    large_test_targets = [
        {"name": "Medium (10K-25K)", "seed_file": "pii-large-samples/conversation-medium.json", "min_chars": 10000, "max_chars": 25000, "target_chars": 15000},
        {"name": "Large (35K+)",     "seed_file": "pii-large-samples/conversation-large.json",  "min_chars": 35000, "max_chars": None,  "target_chars": 38000},
    ]

    def _load_conversation(seed_file):
        with open(repo_root / "validation" / seed_file, "r", encoding="utf-8") as f:
            return json.load(f)

    def _payload_chars(messages):
        return len(json.dumps({"model": model_name, "messages": messages}))

    def _grow_conversation(messages, target_chars):
        """Extend a realistic conversation history until the serialized payload reaches
        target_chars. The single system message is kept once; user/assistant turns are
        repeated with a rotating '[case #n]' tag so PII values stay varied and the body
        remains structurally valid JSON (never a reassembly of redacted fragments)."""
        turns = [m for m in messages if m["role"] != "system"]
        grown = list(messages)
        round_idx = 0
        while _payload_chars(grown) < target_chars and round_idx < 100:
            round_idx += 1
            for m in turns:
                tagged = dict(m)
                tagged["content"] = f"[case #{round_idx}] " + m["content"]
                grown.append(tagged)
        return grown

    if enableLargeRequestTesting:
        for t in large_test_targets:
            seed = _load_conversation(t["seed_file"])
            base_chars = _payload_chars(seed)
            grown = _grow_conversation(seed, t["target_chars"])
            size = _payload_chars(grown)
            # ceil(size / effective_chunk_span); effective span accounts for overlap
            effective_span = max(1, PII_MAX_CHUNK_SIZE - PII_CHUNK_OVERLAP)
            est_chunks = max(1, -(-size // effective_span))
            est_requests = max(1, -(-est_chunks // 5))
            t.update({"messages": grown, "payload_chars": size, "est_chunks": est_chunks, "est_requests": est_requests})
            utils.print_info(f"📄 {t['name']}: seed {base_chars} chars → simulated {size} chars (~{est_chunks} chunk(s), ~{est_requests} Language Service request(s))")
            if size < t["min_chars"] or (t["max_chars"] and size > t["max_chars"]):
                utils.print_error(f"   ⚠️ Simulated size {size} is outside the intended band [{t['min_chars']}, {t['max_chars']}] — adjust target_chars.")
        utils.print_ok("Large request payloads prepared.")
    else:
        utils.print_info("enableLargeRequestTesting is False — set it to True to prepare and run the large-document chunking tests.")


### 🧪 Use Case 3: Large Request / Chunked PII Processing — continuation 11

Exercise 🧪 Use Case 3: Large Request / Chunked PII Processing — continuation 11 and compare the actual response/evidence with the expected feature behavior.

**Input:** `enableLargeRequestTesting`, `large_test_targets`, `large_test_results`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [11]:
with utils.notebook_step('🧪 Use Case 3: Large Request / Chunked PII Processing — continuation 11', inputs={'enableLargeRequestTesting': globals().get('enableLargeRequestTesting'), 'large_test_targets': globals().get('large_test_targets'), 'large_test_results': globals().get('large_test_results')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if enableLargeRequestTesting:
        for target in large_test_targets:
            in_band = target["payload_chars"] >= target["min_chars"] and (
                target["max_chars"] is None or target["payload_chars"] <= target["max_chars"])
            run.check("large/" + target["name"] + "/payload-band", in_band,
                expected=[target["min_chars"], target["max_chars"]], actual=target["payload_chars"])
            large_test_results.append(pii_request(target, "large/" + target["name"], messages=target["messages"]))
    else:
        run.blocked("large-chunked-pii", "Required chunking coverage disabled; enableLargeRequestTesting=True.")


---
## 🚫 Use Case 2: PII Blocking

This use case tests PII blocking where:
- Requests containing PII are rejected with HTTP 400 Bad Request
- Detected PII categories are reported in the error response
- No PII data reaches the backend LLM service
---

<a id='4.1'></a>
### 4️⃣.1 Define PII Blocking Access Contract

Create an access contract that blocks any requests containing PII data.

### 4️⃣.1 Define PII Blocking Access Contract

Prepare the 4️⃣.1 Define PII Blocking Access Contract state shown below for its subsequent feature-specific requests and assertions.

**Input:** `run_id`, `timestamp`

**Output:** `pii_blocking_contract` are available to the following cells.


In [12]:
with utils.notebook_step('4️⃣.1 Define PII Blocking Access Contract', inputs={'run_id': globals().get('run_id'), 'timestamp': globals().get('timestamp')}, expected='`pii_blocking_contract` are available to the following cells.', namespace=globals()):
    # PII Blocking Use Case Configuration
    pii_blocking_contract = {
        "name": f"pii-blocking-contract-{timestamp}",
        "business_unit": "Compliance",
        "use_case_name": "PIIBlocking-" + run_id,
        "environment": "DEV",
        "use_keyvault": False,
        "endpoint_secret": "COMPLIANCE-LLM-ENDPOINT",
        "apikey_secret": "COMPLIANCE-LLM-KEY",
        "description": "Compliance PII Blocking - Reject requests containing PII",
        "pii_config": {
            "mode": "blocking",
            "confidence_threshold": "0.75",
            "entity_exclusions": "PersonType",
            "detection_language": "en"
        }
    }

    utils.print_info(f"PII Blocking Contract Configuration:")
    utils.print_info(f"  Business Unit: {pii_blocking_contract['business_unit']}")
    utils.print_info(f"  Use Case: {pii_blocking_contract['use_case_name']}")
    utils.print_info(f"  PII Mode: {pii_blocking_contract['pii_config']['mode']}")
    utils.print_info(f"  Product ID: LLM-{pii_blocking_contract['business_unit']}-{pii_blocking_contract['use_case_name']}-{pii_blocking_contract['environment']}")


<a id='4.2'></a>
### 4️⃣.2 Create PII Blocking Product Policy

Generate a custom product policy XML that detects PII and blocks requests containing sensitive data.

### 4️⃣.2 Create PII Blocking Product Policy

Construct or inspect the 4️⃣.2 Create PII Blocking Product Policy policy, retaining the feature-specific grants and controls shown below.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** The policy XML/configuration and its visible inspection results; no deployment is hidden in this step.


In [13]:
with utils.notebook_step('4️⃣.2 Create PII Blocking Product Policy', inputs={}, expected='The policy XML/configuration and its visible inspection results; no deployment is hidden in this step.', namespace=globals()):
    pii_blocking_policy = '''<policies>
        <inbound>
            <base />
            <!-- Enable PII Blocking -->
            <set-variable name="piiBlockingEnabled" value="true" />

            <choose>
                <when condition="@(context.Variables.GetValueOrDefault<string>(\"piiBlockingEnabled\") == \"true\")">

                    <!-- Configure PII detection settings -->
                    <set-variable name="piiAnonymizationEnabled" value="true" />
                    <set-variable name="piiConfidenceThreshold" value="0.75" />
                    <set-variable name="piiEntityCategoryExclusions" value="PersonType" />
                    <set-variable name="piiDetectionLanguage" value="en" />

                    <!-- Configure custom regex patterns for additional PII detection -->
                    <set-variable name="piiRegexPatterns" value="@{
                        var patterns = new JArray {
                            new JObject {
                                [\"pattern\"] = @\"\\b\\d{4}[- ]?\\d{4}[- ]?\\d{4}[- ]?\\d{4}\\b\",
                                [\"category\"] = \"CREDIT_CARD\"
                            },
                            new JObject {
                                [\"pattern\"] = @\"\\b[A-Z]{2}\\d{6}[A-Z]\\b\",
                                [\"category\"] = \"PASSPORT_NUMBER\"
                            },
                            new JObject {
                                [\"pattern\"] = @\"\\b784-\\d{4}-\\d{7}-\\d{1}\\b\",
                                [\"category\"] = \"EMIRATES_ID\"
                            }
                        };
                        return patterns.ToString();
                    }" />

                    <!-- Capture request body for PII processing -->
                    <set-variable name="piiInputContent" value="@(context.Request.Body.As<string>(preserveContent: true))" />

                    <!-- Apply PII anonymization to detect PII -->
                    <include-fragment fragment-id="pii-anonymization" />

                    <!-- Check if any PII was detected (piiMappings will have entries if PII found) -->
                    <choose>
                        <when condition="@{
                            var mappings = context.Variables.GetValueOrDefault<string>(\"piiMappings\", \"[]\");
                            var mappingsArray = JArray.Parse(mappings);
                            return mappingsArray.Count > 0;
                        }">
                            <return-response>
                                <set-status code="400" reason="Bad Request" />
                                <set-header name="Content-Type" exists-action="override">
                                    <value>application/json</value>
                                </set-header>
                                <set-body>@{
                                    var mappings = JArray.Parse(context.Variables.GetValueOrDefault<string>("piiMappings", "[]"));
                                    var categories = new HashSet<string>();
                                    foreach (var mapping in mappings) {
                                        var placeholder = mapping["placeholder"].ToString();
                                        var category = placeholder.TrimStart('<').Split('_')[0];
                                        categories.Add(category);
                                    }
                                    return new JObject(
                                        new JProperty("error", new JObject(
                                            new JProperty("code", "PII_DETECTED"),
                                            new JProperty("message", "Request blocked: Personal Identifiable Information (PII) detected in the request."),
                                            new JProperty("detectedCategories", string.Join(", ", categories)),
                                            new JProperty("entityCount", mappings.Count)
                                        ))
                                    ).ToString();
                                }</set-body>
                            </return-response>
                        </when>
                    </choose>
                </when>
            </choose>
        </inbound>
        <backend>
            <base />
        </backend>
        <outbound>
            <base />
        </outbound>
        <on-error>
            <base />
        </on-error>
    </policies>'''
    pii_blocking_policy = policy_xml(pii_blocking_policy)
    utils.format_output('4️⃣.2 Create PII Blocking Product Policy', {'pii_blocking_policy': pii_blocking_policy}, status='info', part='RESULT')


<a id='4.3'></a>
### 4️⃣.3 Create Parameter File and Deploy PII Blocking Contract

### 4️⃣.3 Create Parameter File and Deploy PII Blocking Contract

Generate the editable parameter artifact for 4️⃣.3 Create Parameter File and Deploy PII Blocking Contract and display the proposed configuration without deploying it.

**Input:** `pii_blocking_contract`, `pii_blocking_policy`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [14]:
with utils.notebook_step('4️⃣.3 Create Parameter File and Deploy PII Blocking Contract', inputs={'pii_blocking_contract': globals().get('pii_blocking_contract'), 'pii_blocking_policy': globals().get('pii_blocking_policy'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    blocking_parameters = access_parameters(context, apimClientTool, pii_blocking_contract, pii_blocking_policy,
        key_vault=None)

    blocking_params_file = write_parameters(
        repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
        blocking_parameters, run_dir / "blocking")
    utils.format_output('4️⃣.3 Create Parameter File and Deploy PII Blocking Contract', {'value_1': preview(blocking_params_file)}, status='info', part='RESULT')


<a id='4.4'></a>
### 4️⃣.4 Retrieve API Key for PII Blocking Contract

### 4️⃣.4 Retrieve API Key for PII Blocking Contract

Deploy the reviewed artifact for 4️⃣.4 Retrieve API Key for PII Blocking Contract, then inspect its real resource outputs and ownership.

**Input:** `blocking_params_file`, `location`, `preexisting_products`, `owned_products`, `pii_blocking_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [15]:
with utils.notebook_step('4️⃣.4 Retrieve API Key for PII Blocking Contract', inputs={'blocking_params_file': globals().get('blocking_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'pii_blocking_contract': globals().get('pii_blocking_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    blocking_outputs = deploy_contract(blocking_params_file, location, pii_blocking_contract["name"],
        required_outputs=("products", "subscriptions"))
    blocking_product_id, blocking_subscription_name = contract_identity(
        blocking_outputs, preexisting_products, owned_products)
    blocking_api_key = apimClientTool.get_subscription_key(blocking_subscription_name)


<a id='4.5'></a>
### 4️⃣.5 Test PII Blocking with Various PII Types

Send test requests containing PII data to verify they are blocked, and requests without PII are allowed.

### 4️⃣.5 Test PII Blocking with Various PII Types

Prepare the 4️⃣.5 Test PII Blocking with Various PII Types state shown below for its subsequent feature-specific requests and assertions.

**Input:** `model_name`, `t`

**Output:** `pii_blocking_test_payloads` are available to the following cells.


In [16]:
with utils.notebook_step('4️⃣.5 Test PII Blocking with Various PII Types', inputs={'model_name': globals().get('model_name'), 't': globals().get('t')}, expected='`pii_blocking_test_payloads` are available to the following cells.', namespace=globals()):
    # Define test payloads for PII Blocking - mix of requests with and without PII
    pii_blocking_test_payloads = [
        {
            "name": "Request WITH PII - Should be BLOCKED",
            "description": "Request containing email and name - should return 400",
            "should_block": True,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": "Send an email to John Smith at john.smith@company.com about the project update."}
                ]
            },
            "expected_pii": ["Person", "Email"]
        },
        {
            "name": "Request WITHOUT PII - Should be ALLOWED",
            "description": "Generic request without any PII - should return 200",
            "should_block": False,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": "What is the capital of France?"}
                ]
            },
            "expected_pii": []
        },
        {
            "name": "Request WITH Credit Card - Should be BLOCKED",
            "description": "Request containing credit card number - should return 400",
            "should_block": True,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": "My card number is 4532-1234-5678-9012. Why was I charged twice?"}
                ]
            },
            "expected_pii": ["CREDIT_CARD"]
        },
        {
            "name": "Request WITHOUT PII - Technical Question",
            "description": "Technical question without PII - should return 200",
            "should_block": False,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful coding assistant."},
                    {"role": "user", "content": "How do I create a REST API in Python using Flask?"}
                ]
            },
            "expected_pii": []
        },
        {
            "name": "Request WITH Phone Number - Should be BLOCKED",
            "description": "Request containing phone number - should return 400",
            "should_block": True,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": "Please call me back at +1-555-123-4567 to discuss the issue."}
                ]
            },
            "expected_pii": ["PhoneNumber"]
        },
        {
            "name": "Request WITH Emirates ID - Should be BLOCKED",
            "description": "Request containing Emirates ID (custom regex) - should return 400",
            "should_block": True,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": "My Emirates ID is 784-1990-1234567-1. Can you help me with my visa application?"}
                ]
            },
            "expected_pii": ["EMIRATES_ID"]
        },
        {
            "name": "Request WITHOUT PII - Math Question",
            "description": "Simple math question without PII - should return 200",
            "should_block": False,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful math tutor."},
                    {"role": "user", "content": "What is 25 multiplied by 4?"}
                ]
            },
            "expected_pii": []
        },
        {
            "name": "Request WITH Multiple PII Types - Should be BLOCKED",
            "description": "Request with multiple PII types - should return 400 with all categories",
            "should_block": True,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": "Hi, I'm Sarah Johnson (sarah.j@email.com, +971-50-987-6543). My IBAN is AE070331234567890123456. Please help me."}
                ]
            },
            "expected_pii": ["Person", "Email", "PhoneNumber", "InternationalBankingAccountNumber"]
        }
    ]

    utils.print_info(f"\nDefined {len(pii_blocking_test_payloads)} test payloads for PII Blocking tests")
    utils.print_info(f"  - Requests that should be BLOCKED: {sum(1 for t in pii_blocking_test_payloads if t['should_block'])}")
    utils.print_info(f"  - Requests that should be ALLOWED: {sum(1 for t in pii_blocking_test_payloads if not t['should_block'])}")


### 4️⃣.5 Test PII Blocking with Various PII Types — continuation 17

Verify that the gateway detects PII and blocks the request: HTTP 400, error code `PII_DETECTED`, and a positive integer `entityCount`. The response reports category labels (for example `Person`, `CreditCard` or `CREDIT`), not full placeholders. Exact category-name matches are not a pass/fail requirement; returned labels and counts remain visible as diagnostic evidence. This validates blocking/detection, not exhaustive classification accuracy.

**Input:** `pii_blocking_test_payloads`, `chat_completions_url`, `test`, `response`, `data`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [17]:
with utils.notebook_step('4️⃣.5 Test PII Blocking with Various PII Types — continuation 17', inputs={'pii_blocking_test_payloads': globals().get('pii_blocking_test_payloads'), 'chat_completions_url': globals().get('chat_completions_url'), 'test': globals().get('test'), 'response': globals().get('response'), 'data': globals().get('data')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    blocking_test_results = []
    for test in pii_blocking_test_payloads:
        response = transport.request("POST", chat_completions_url,
            headers={"api-key": blocking_api_key}, json=test["payload"])
        data = response.json_body if isinstance(response.json_body, dict) else {}
        error = data.get("error") if isinstance(data.get("error"), dict) else {}
        categories = error.get("detectedCategories")
        entity_count = error.get("entityCount")
        if test["should_block"]:
            passed = (response.status_code == 400 and error.get("code") == "PII_DETECTED"
                      and isinstance(entity_count, int) and not isinstance(entity_count, bool)
                      and entity_count > 0)
        else:
            passed = response.status_code == 200 and bool(completion_text(data).strip())
        passed = bool(passed and not response.error)
        run.check("blocking/" + test["name"], passed,
            expected="400 PII_DETECTED with a positive detected-entity count" if test["should_block"] else "200 nonempty completion",
            actual=response.status_code, error_code=error.get("code"), entity_count=entity_count,
            detected_categories=categories, transport_error=response.error, duration=response.duration)
        blocking_test_results.append({"test_name": test["name"], "status_code": response.status_code,
            "test_passed": passed, "success": passed, "elapsed_time": response.duration})


---
## 📈 Use Case 3: PII Processing Performance Analytics

This section pulls **recent PII processing telemetry** from the gateway's Cosmos DB
(`pii-usage-container`) and provides:

- 🔢 Quantitative report (records, entity totals, success rates, content size impact)
- 📊 Visualizations (entity-category mix, anonymization vs. deanonymization performance, content-length transformation, processing volume over time)
- 🤖 Optional **LLM-powered reasoning** over the analytics summary, sent through the **PII Masking Access Contract** so any residual PII in the summary is anonymized before reaching the model.

> Make sure the gateway is configured to log PII processing events to a Cosmos DB container with documents shaped like the `PII_Processing` schema produced by the `pii-state-saving` policy fragment.
---

<a id='5.1'></a>
### 5️⃣.1 Configure Cosmos DB Connection

Configure the Cosmos DB account, database, and container that store the PII processing
telemetry, plus how many recent records to load.

Authentication options:
- `aad`  → uses your Azure AD identity via `DefaultAzureCredential` (recommended; requires the *Cosmos DB Built-in Data Reader* role on the account)
- `key`  → uses the provided account key (set `cosmos_account_key`)

### 5️⃣.1 Configure Cosmos DB Connection

Query or inspect 5️⃣.1 Configure Cosmos DB Connection evidence for this run; configuration and HTTP success alone are not evidence.

**Input:** `enable_cosmos_analytics`, `filter_product_id`, `filter_subscription_id`, `cosmos_account_endpoint`, `cosmos_database_name`

**Output:** Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.


In [18]:
with utils.notebook_step('5️⃣.1 Configure Cosmos DB Connection', inputs={'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'filter_product_id': globals().get('filter_product_id'), 'filter_subscription_id': globals().get('filter_subscription_id'), 'cosmos_account_endpoint': globals().get('cosmos_account_endpoint'), 'cosmos_database_name': globals().get('cosmos_database_name')}, expected='Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.', namespace=globals()):
    if enable_cosmos_analytics:
        utils.print_info(f"Cosmos endpoint     : {cosmos_account_endpoint}")
        utils.print_info(f"Database / container: {cosmos_database_name} / {cosmos_container_name}")
        utils.print_info(f"Auth mode           : {cosmos_auth_mode}")
        utils.print_info(f"Top N records       : {top_n_records}")
        if filter_product_id:
            utils.print_info(f"Filter product      : {filter_product_id}")
        if filter_subscription_id:
            utils.print_info(f"Filter subscription : {filter_subscription_id}")
    else:
        run.skip("cosmos-section-36", "Optional analytics/permission grant disabled.")


<a id='5.1.1'></a>
### 5️⃣.1.1 Grant Current User Cosmos DB Data-Plane Access

Cosmos DB **control-plane** roles (Owner, Contributor) do **not** grant data-plane (SQL) read/write access. To query items via AAD (`DefaultAzureCredential`), the signed-in user needs a SQL role assignment.

This step uses the Azure CLI to assign the **Cosmos DB Built-in Data Reader** role (`00000000-0000-0000-0000-000000000001`) at the account scope (`/`). Switch to `00000000-0000-0000-0000-000000000002` (Built-in Data Contributor) if you also need writes.

**Prerequisites for the caller:**
- `az login` already done (the same identity that will run the queries).
- Permission `Microsoft.DocumentDB/databaseAccounts/sqlRoleAssignments/write` on the Cosmos account (e.g. *Cosmos DB Operator*, *Owner*, or *User Access Administrator*). Otherwise the cell prints the exact CLI command an admin can run on your behalf.

Role propagation can take **1–5 minutes** — if the next cell still 401s, wait briefly and re-run.

>**NOTE**: if you get `Continuous access evaluation resulted in challenge`, try to run the following `az logout`, `az account clear` and `az login` again to refresh your credentials and ensure the new role assignment is recognized.


### 5️⃣.1.1 Grant Current User Cosmos DB Data-Plane Access

Query or inspect 5️⃣.1.1 Grant Current User Cosmos DB Data-Plane Access evidence for this run; configuration and HTTP success alone are not evidence.

**Input:** `enable_cosmos_analytics`, `grant_cosmos_data_access`, `governance_hub_resource_group`, `cosmos_account_endpoint`

**Output:** Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.


In [19]:
with utils.notebook_step('5️⃣.1.1 Grant Current User Cosmos DB Data-Plane Access', inputs={'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'grant_cosmos_data_access': globals().get('grant_cosmos_data_access'), 'governance_hub_resource_group': globals().get('governance_hub_resource_group'), 'cosmos_account_endpoint': globals().get('cosmos_account_endpoint')}, expected='Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.', namespace=globals()):
    if enable_cosmos_analytics and grant_cosmos_data_access:
        # ============================================================================
        # 🔐 GRANT CURRENT USER COSMOS DB SQL DATA-PLANE ACCESS (via Azure CLI)
        # ----------------------------------------------------------------------------
        # Assigns the Cosmos DB Built-in Data Reader role at account scope to the
        # currently signed-in AAD principal so the notebook can query items via
        # DefaultAzureCredential.
        # ============================================================================
        from urllib.parse import urlparse
        import json as _json

        # --- Resolve account name from endpoint and pick resource group --------------
        cosmos_account_name = urlparse(cosmos_account_endpoint).hostname.split('.')[0]
        cosmos_resource_group = governance_hub_resource_group  # default to hub RG

        # Built-in role definition IDs (well-known, same in every Cosmos account)
        COSMOS_DATA_READER       = '00000000-0000-0000-0000-000000000001'
        COSMOS_DATA_CONTRIBUTOR  = '00000000-0000-0000-0000-000000000002'
        cosmos_data_role_id = COSMOS_DATA_READER  # change to COSMOS_DATA_CONTRIBUTOR if writes are needed

        utils.print_info(f"Cosmos account : {cosmos_account_name}")
        utils.print_info(f"Resource group : {governance_hub_resource_group}")
        utils.print_info(f"Role definition: {cosmos_data_role_id} (Data Reader)")

        # --- Resolve signed-in principal objectId ------------------------------------
        principal_object_id = None
        who = utils.run("az ad signed-in-user show --query id -o tsv",
                        'Resolved signed-in user', 'Failed to resolve signed-in user')
        if who.success:
            principal_object_id = (who.text or '').strip()

        if not principal_object_id:
            utils.print_error('Could not resolve signed-in user objectId. Run `az login` and re-run this cell.')
        else:
            utils.print_info(f"Principal id   : {principal_object_id}")

            # --- Idempotency: skip if an assignment for this principal already exists -
            list_cmd = (f"az cosmosdb sql role assignment list "
                        f"--account-name {cosmos_account_name} "
                        f"--resource-group {governance_hub_resource_group} -o json")
            existing = utils.run(list_cmd, 'Listed existing role assignments',
                                 'Failed to list role assignments (need Cosmos DB Operator/Owner)')
            already_assigned = False
            if existing.success and existing.text:
                try:
                    for a in _json.loads(existing.text):
                        if (a.get('principalId','').lower() == principal_object_id.lower()
                                and a.get('roleDefinitionId','').endswith(cosmos_data_role_id)):
                            already_assigned = True
                            break
                except Exception as ex:
                    utils.print_error(f"Could not parse existing assignments: {ex}")

            create_cmd = (f"az cosmosdb sql role assignment create "
                          f"--account-name {cosmos_account_name} "
                          f"--resource-group {governance_hub_resource_group} "
                          f"--scope \"/\" "
                          f"--principal-id {principal_object_id} "
                          f"--role-definition-id {cosmos_data_role_id}")

            if already_assigned:
                utils.print_ok('Role assignment already exists — nothing to do.')
            else:
                result = utils.run(create_cmd, 'Granted Cosmos DB data-plane access',
                                   'Failed to create role assignment')
                if not result.success:
                    utils.print_info('If you lack permission, ask an admin to run:')
                    utils.format_output('5️⃣.1.1 Grant Current User Cosmos DB Data-Plane Access', {'value_1': '  ' + create_cmd}, status='info', part='RESULT')
                else:
                    utils.print_info('Role propagation can take 1–5 minutes before queries succeed.')
    else:
        run.skip("cosmos-section-38", "Optional analytics/permission grant disabled.")


<a id='5.2'></a>
### 5️⃣.2 Query Recent PII Processing Records

Loads the most recent `top_n_records` PII processing documents from Cosmos DB and
flattens them into a `pandas` DataFrame for analysis. `azure-cosmos` and `pandas` are installed and verified by [Validation Environment Setup](citadel-environment-setup.ipynb); this cell does not change the environment.

### 5️⃣.2 Query Recent PII Processing Records

Prepare the 5️⃣.2 Query Recent PII Processing Records state shown below for its subsequent feature-specific requests and assertions.

**Input:** `enable_cosmos_analytics`, `filter_product_id`, `filter_subscription_id`, `cosmos_database_name`, `cosmos_container_name`, `cosmos_account_endpoint`

**Output:** `cosmos_client`, `cosmos_db`, `cosmos_container`, `where_clauses`, `parameters`, `query` are available to the following cells.


In [23]:
with utils.notebook_step('5️⃣.2 Query Recent PII Processing Records', inputs={'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'filter_product_id': globals().get('filter_product_id'), 'filter_subscription_id': globals().get('filter_subscription_id'), 'cosmos_database_name': globals().get('cosmos_database_name'), 'cosmos_container_name': globals().get('cosmos_container_name'), 'cosmos_account_endpoint': globals().get('cosmos_account_endpoint')}, expected='`cosmos_client`, `cosmos_db`, `cosmos_container`, `where_clauses`, `parameters`, `query` are available to the following cells.', namespace=globals()):
    if enable_cosmos_analytics:
        from azure.cosmos import CosmosClient

        import pandas as pd
        from datetime import datetime, timezone

        # Build Cosmos client
        if cosmos_auth_mode.lower() == "aad":
            from azure.identity import DefaultAzureCredential
            cosmos_client = CosmosClient(cosmos_account_endpoint, credential=DefaultAzureCredential())
        else:
            if not cosmos_account_key:
                raise ValueError("cosmos_account_key is empty but cosmos_auth_mode == 'key'")
            cosmos_client = CosmosClient(cosmos_account_endpoint, credential=cosmos_account_key)

        cosmos_db        = cosmos_client.get_database_client(cosmos_database_name)
        cosmos_container = cosmos_db.get_container_client(cosmos_container_name)

        # Build query (only PII_Processing docs)
        where_clauses = ["c.type = 'PII_Processing'"]
        parameters    = []
        if filter_product_id:
            where_clauses.append("c.context.product.id = @prodId")
            parameters.append({"name": "@prodId", "value": filter_product_id})
        if filter_subscription_id:
            where_clauses.append("c.context.subscription.id = @subId")
            parameters.append({"name": "@subId", "value": filter_subscription_id})

        query = (
            f"SELECT TOP {int(top_n_records)} * FROM c "
            f"WHERE {' AND '.join(where_clauses)} "
            f"ORDER BY c._ts DESC"
        )
        utils.print_info(f"Cosmos query: {query}")

        start_time = time.time()
        pii_records = list(cosmos_container.query_items(
            query=query,
            parameters=parameters,
            enable_cross_partition_query=True
        ))
        elapsed_time = time.time() - start_time
        utils.print_ok(f"✅ Retrieved {len(pii_records)} record(s) in {elapsed_time:.2f}s")

        # Flatten into a DataFrame
        def _flatten(rec):
            ctx       = rec.get("context", {}) or {}
            api       = ctx.get("api", {}) or {}
            product   = ctx.get("product", {}) or {}
            sub       = ctx.get("subscription", {}) or {}
            process   = rec.get("process", {}) or {}
            metrics   = rec.get("metrics", {}) or {}
            flow      = rec.get("processingFlow", {}) or {}
            deanon    = rec.get("deanonymization", {}) or {}
            cats      = rec.get("entityCategories", {}) or {}
            return {
                "id"                       : rec.get("id"),
                "timestamp"                : rec.get("timestamp"),
                "ts"                       : datetime.fromtimestamp(rec.get("_ts", 0), tz=timezone.utc) if rec.get("_ts") else None,
                "targetDeployment"         : rec.get("targetDeployment"),
                "api_id"                   : api.get("id"),
                "operation"                : (api.get("operation") or {}).get("id"),
                "product_id"               : product.get("id"),
                "subscription_id"          : sub.get("id"),
                "anonymization_enabled"    : process.get("anonymizationEnabled"),
                "confidence_threshold"     : process.get("confidenceThreshold"),
                "entity_exclusions"        : process.get("entityCategoryExclusions"),
                "deanonymization_performed": process.get("deanonymizationPerformed"),
                "entity_count"             : rec.get("entityCount", 0) or 0,
                "entity_categories"        : cats,
                "input_len"                : metrics.get("inputContentLength", 0) or 0,
                "anonymized_len"           : metrics.get("anonymizedContentLength", 0) or 0,
                "deanon_input_len"         : metrics.get("deanonymizeInputLength", 0) or 0,
                "deanon_output_len"        : metrics.get("deanonymizedOutputLength", 0) or 0,
                "anon_performed"           : (flow.get("anonymization") or {}).get("performed", False),
                "deanon_performed"         : (flow.get("deanonymization") or {}).get("performed", False),
                "deanon_success"           : (flow.get("deanonymization") or {}).get("success", False),
                "deanon_status"            : deanon.get("status"),
                "deanon_input_source"      : deanon.get("inputSource"),
            }

        pii_df = pd.DataFrame([_flatten(r) for r in pii_records])
        if not pii_df.empty:
            pii_df["ts"] = pd.to_datetime(pii_df["ts"], utc=True, errors="coerce")
            pii_df.sort_values("ts", inplace=True)

        utils.print_info(f"Loaded DataFrame shape: {pii_df.shape}")
        pii_df.head()
    else:
        run.skip("cosmos-section-40", "Optional analytics/permission grant disabled.")


CosmosHttpResponseError: (Forbidden) Request originated from IP 94.204.156.50 through public internet. This is blocked by your Cosmos DB account firewall settings. More info: https://aka.ms/cosmosdb-tsg-forbidden
ActivityId: 77407271-1fcd-486f-88d2-7dfe48c68071, Microsoft.Azure.Documents.Common/2.14.0
Code: Forbidden
Message: Request originated from IP 94.204.156.50 through public internet. This is blocked by your Cosmos DB account firewall settings. More info: https://aka.ms/cosmosdb-tsg-forbidden
ActivityId: 77407271-1fcd-486f-88d2-7dfe48c68071, Microsoft.Azure.Documents.Common/2.14.0 , Endpoint: https://cosmos-ttg2fgnpni24u.documents.azure.com

<a id='5.3'></a>
### 5️⃣.3 Quantitative Report & Visualizations

Builds a quantitative summary of how the gateway performed on PII detection,
anonymization, and deanonymization, then renders charts highlighting:

1. Detected entities by category
2. Anonymization vs. deanonymization performance
3. Content-length impact (original → anonymized → deanonymized)
4. Volume of PII processing over time

### 5️⃣.3 Quantitative Report & Visualizations

Prepare the 5️⃣.3 Quantitative Report & Visualizations state shown below for its subsequent feature-specific requests and assertions.

**Input:** `enable_cosmos_analytics`, `pii_df`, `cats`

**Output:** `pii_analytics_summary`, `total_records`, `total_entities`, `avg_entities`, `records_with_pii`, `anon_performed_rate` are available to the following cells.


In [ ]:
with utils.notebook_step('5️⃣.3 Quantitative Report & Visualizations', inputs={'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'pii_df': globals().get('pii_df'), 'cats': globals().get('cats')}, expected='`pii_analytics_summary`, `total_records`, `total_entities`, `avg_entities`, `records_with_pii`, `anon_performed_rate` are available to the following cells.', namespace=globals()):
    if enable_cosmos_analytics:
        import matplotlib.pyplot as plt
        from collections import Counter

        pii_analytics_summary = {}

        if pii_df.empty:
            utils.print_error("No PII processing records returned — skipping analytics.")
        else:
            # ---------- Quantitative report ----------
            total_records          = len(pii_df)
            total_entities         = int(pii_df["entity_count"].sum())
            avg_entities           = float(pii_df["entity_count"].mean())
            records_with_pii       = int((pii_df["entity_count"] > 0).sum())
            anon_performed_rate    = float(pii_df["anon_performed"].mean()) * 100 if total_records else 0
            deanon_performed_rate  = float(pii_df["deanon_performed"].mean()) * 100 if total_records else 0
            deanon_success_rate    = (
                float(pii_df.loc[pii_df["deanon_performed"], "deanon_success"].mean()) * 100
                if pii_df["deanon_performed"].any() else 0
            )

            # Aggregate entity categories across all records
            category_counter = Counter()
            for cats in pii_df["entity_categories"]:
                if isinstance(cats, dict):
                    category_counter.update({k: int(v) for k, v in cats.items()})

            # Length deltas
            length_df = pii_df[["input_len", "anonymized_len", "deanon_input_len", "deanon_output_len"]].copy()
            avg_anon_delta_pct = (
                ((length_df["anonymized_len"] - length_df["input_len"]) / length_df["input_len"].replace(0, pd.NA))
                .dropna().mean() * 100
            ) if not length_df.empty else 0
            avg_deanon_delta_pct = (
                ((length_df["deanon_output_len"] - length_df["deanon_input_len"]) / length_df["deanon_input_len"].replace(0, pd.NA))
                .dropna().mean() * 100
            ) if not length_df.empty else 0

            pii_analytics_summary = {
                "total_records"           : total_records,
                "records_with_pii"        : records_with_pii,
                "records_without_pii"     : total_records - records_with_pii,
                "total_entities_detected" : total_entities,
                "avg_entities_per_record" : round(avg_entities, 2),
                "anonymization_performed_rate_pct"  : round(anon_performed_rate, 2),
                "deanonymization_performed_rate_pct": round(deanon_performed_rate, 2),
                "deanonymization_success_rate_pct"  : round(deanon_success_rate, 2),
                "avg_content_length_change_anon_pct"  : round(float(avg_anon_delta_pct or 0), 2),
                "avg_content_length_change_deanon_pct": round(float(avg_deanon_delta_pct or 0), 2),
                "entity_category_counts"  : dict(category_counter.most_common()),
                "top_products"            : pii_df["product_id"].value_counts().head(5).to_dict(),
                "top_subscriptions"       : pii_df["subscription_id"].value_counts().head(5).to_dict(),
                "time_window_utc"         : {
                    "from": str(pii_df["ts"].min()),
                    "to"  : str(pii_df["ts"].max()),
                },
            }

            utils.print_info(f"\n{'='*80}")
            utils.print_info(f"📊 PII PROCESSING QUANTITATIVE REPORT (top {total_records} records)")
            utils.print_info(f"{'='*80}")
            utils.print_info(f"Records analyzed              : {total_records}")
            utils.print_info(f"Records containing PII        : {records_with_pii} ({records_with_pii/total_records*100:.1f}%)")
            utils.print_info(f"Total entities detected       : {total_entities}")
            utils.print_info(f"Avg entities / record         : {avg_entities:.2f}")
            utils.print_info(f"Anonymization performed rate  : {anon_performed_rate:.1f}%")
            utils.print_info(f"Deanonymization performed rate: {deanon_performed_rate:.1f}%")
            utils.print_ok  (f"Deanonymization success rate  : {deanon_success_rate:.1f}%")
            utils.print_info(f"Avg content size change (anon)   : {pii_analytics_summary['avg_content_length_change_anon_pct']}%")
            utils.print_info(f"Avg content size change (deanon) : {pii_analytics_summary['avg_content_length_change_deanon_pct']}%")
            utils.print_info(f"Top categories                : {dict(category_counter.most_common(5))}")

            # ---------- Visualizations ----------
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            fig.suptitle("PII Processing Performance — Gateway Telemetry", fontsize=14, fontweight="bold")

            # 1) Entities by category
            ax = axes[0, 0]
            if category_counter:
                cats_sorted = category_counter.most_common()
                ax.barh([c for c, _ in cats_sorted][::-1],
                        [v for _, v in cats_sorted][::-1],
                        color="#4C72B0")
                ax.set_title("Detected entities by category")
                ax.set_xlabel("Count")
            else:
                ax.text(0.5, 0.5, "No entity categories detected", ha="center", va="center")
                ax.set_axis_off()

            # 2) Anonymization vs Deanonymization performance
            ax = axes[0, 1]
            perf_labels = ["Records w/ PII", "Anon performed", "Deanon performed", "Deanon success"]
            perf_values = [
                records_with_pii,
                int(pii_df["anon_performed"].sum()),
                int(pii_df["deanon_performed"].sum()),
                int(pii_df.loc[pii_df["deanon_performed"], "deanon_success"].sum()),
            ]
            bars = ax.bar(perf_labels, perf_values, color=["#55A868", "#4C72B0", "#8172B2", "#C44E52"])
            ax.set_title("Anonymization / Deanonymization performance")
            ax.set_ylabel("Records")
            for b, v in zip(bars, perf_values):
                ax.text(b.get_x() + b.get_width() / 2, b.get_height(), str(v),
                        ha="center", va="bottom", fontsize=9)
            ax.tick_params(axis="x", rotation=20)

            # 3) Content length transformation (avg)
            ax = axes[1, 0]
            avg_lengths = [
                length_df["input_len"].mean(),
                length_df["anonymized_len"].mean(),
                length_df["deanon_input_len"].mean(),
                length_df["deanon_output_len"].mean(),
            ]
            ax.bar(["Input", "Anonymized", "Deanon Input", "Deanon Output"],
                   avg_lengths, color="#937860")
            ax.set_title("Avg content length through PII pipeline")
            ax.set_ylabel("Characters")

            # 4) Volume over time
            ax = axes[1, 1]
            if pii_df["ts"].notna().any():
                per_minute = (pii_df.set_index("ts")
                                    .assign(n=1)["n"]
                                    .resample("1min").sum())
                ax.plot(per_minute.index, per_minute.values, marker="o", color="#4C72B0")
                ax.set_title("PII processing volume over time")
                ax.set_ylabel("Records / min")
                ax.tick_params(axis="x", rotation=30)
            else:
                ax.text(0.5, 0.5, "No timestamps available", ha="center", va="center")
                ax.set_axis_off()

            plt.tight_layout(rect=[0, 0, 1, 0.96])
            plt.show()
    else:
        run.skip("cosmos-section-42", "Optional analytics/permission grant disabled.")


<a id='5.4'></a>
### 5️⃣.4 Define & Deploy PII Analytics Access Contract

The previously created **PII Masking** and **PII Blocking** contracts have PII processing
policies enabled — sending the analytics payload (which itself contains anonymized PII
samples and entity categories) through them would re-trigger detection, double-mask
values, and pollute the telemetry we are analyzing.

We therefore provision a **separate access contract** dedicated to PII analytics
reasoning. It has:

- **No PII processing fragments** in its product policy.
- **Configurable model-level RBAC** via the `allowed_models` list.
- **Configurable capacity allocation** via `tokens_per_minute`, `token_quota`,
  and `token_quota_period`.


### 5️⃣.4 Define & Deploy PII Analytics Access Contract

Query or inspect 5️⃣.4 Define & Deploy PII Analytics Access Contract evidence for this run; configuration and HTTP success alone are not evidence.

**Input:** `enable_cosmos_analytics`, `timestamp`, `run_id`, `model_name`

**Output:** Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.


In [ ]:
with utils.notebook_step('5️⃣.4 Define & Deploy PII Analytics Access Contract', inputs={'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'timestamp': globals().get('timestamp'), 'run_id': globals().get('run_id'), 'model_name': globals().get('model_name')}, expected='Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.', namespace=globals()):
    if enable_cosmos_analytics:
        # ============================================================================
        # 📋 PII ANALYTICS ACCESS CONTRACT — configurable model RBAC + capacity
        # ============================================================================
        # Reuse the timestamp from the earlier contract definitions when available so
        # all three contracts deployed by this notebook share the same suffix.
        try:
            timestamp
        except NameError:
            timestamp = run_id

        pii_analytics_contract = {
            "name": f"pii-analytics-contract-{timestamp}",
            "business_unit": "HR",
            "use_case_name": "PIIAnalytics-" + run_id,
            "environment": "DEV",
            "use_keyvault": False,
            "endpoint_secret": "HR-PII-LLM-ENDPOINT",
            "apikey_secret":   "HR-PII-LLM-KEY",
            "description": "HR PII Analytics - LLM reasoning over PII processing telemetry (no PII fragments)",

            # 🔐 Model-level RBAC — list of model names allowed through this contract
            "allowed_models":      [model_name],

            # 📊 Capacity allocation
            "tokens_per_minute":   20000,        # TPM throttle
            "token_quota":         500000,       # long-term token budget
            "token_quota_period": "Monthly",     # Hourly | Daily | Weekly | Monthly
        }

        utils.print_info("PII Analytics Contract Configuration:")
        utils.print_info(f"  Business Unit  : {pii_analytics_contract['business_unit']}")
        utils.print_info(f"  Use Case       : {pii_analytics_contract['use_case_name']}")
        utils.print_info(f"  Allowed Models : {pii_analytics_contract['allowed_models']}")
        utils.print_info(f"  TPM Limit      : {pii_analytics_contract['tokens_per_minute']:,}")
        utils.print_info(f"  Token Quota    : {pii_analytics_contract['token_quota']:,} per {pii_analytics_contract['token_quota_period']}")
        utils.print_info(f"  Product ID     : LLM-{pii_analytics_contract['business_unit']}-{pii_analytics_contract['use_case_name']}-{pii_analytics_contract['environment']}")
    else:
        run.skip("cosmos-section-44", "Optional analytics/permission grant disabled.")


<a id='5.4.2'></a>
#### 5️⃣.4.2 Build Analytics Product Policy (Model RBAC + Capacity)

The policy uses two well-known APIM fragments shipped with this accelerator:

- `set-llm-requested-model` — extracts the requested model name from the inbound payload.
- `validate-model-access` — denies the request if the requested model is **not** in the
  `allowedModels` CSV variable.

and the built-in `llm-token-limit` policy to enforce the per-minute and quota limits.


### 5️⃣.4.2 Build Analytics Product Policy (Model RBAC + Capacity)

Query or inspect 5️⃣.4.2 Build Analytics Product Policy (Model RBAC + Capacity) evidence for this run; configuration and HTTP success alone are not evidence.

**Input:** `enable_cosmos_analytics`, `pii_analytics_contract`

**Output:** Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.


In [ ]:
with utils.notebook_step('5️⃣.4.2 Build Analytics Product Policy (Model RBAC + Capacity)', inputs={'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'pii_analytics_contract': globals().get('pii_analytics_contract')}, expected='Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.', namespace=globals()):
    if enable_cosmos_analytics:
        contract = pii_analytics_contract
        pii_analytics_policy = access_policy(contract["allowed_models"], contract["tokens_per_minute"],
            contract["token_quota"], contract["token_quota_period"],
            inbound_extra='<set-variable name="enableResponseHeaders" value="@(true)" />')
        utils.format_output('5️⃣.4.2 Build Analytics Product Policy (Model RBAC + Capacity)', {'pii_analytics_policy': pii_analytics_policy}, status='info', part='RESULT')
    else:
        run.skip("cosmos-section-46", "Optional analytics/permission grant disabled.")


<a id='5.4.3'></a>
#### 5️⃣.4.3 Generate Parameter File and Deploy Analytics Contract


### 5️⃣.4.3 Generate Parameter File and Deploy Analytics Contract

Generate the editable parameter artifact for 5️⃣.4.3 Generate Parameter File and Deploy Analytics Contract and display the proposed configuration without deploying it.

**Input:** `enable_cosmos_analytics`, `pii_analytics_contract`, `pii_analytics_policy`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('5️⃣.4.3 Generate Parameter File and Deploy Analytics Contract', inputs={'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'pii_analytics_contract': globals().get('pii_analytics_contract'), 'pii_analytics_policy': globals().get('pii_analytics_policy'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    if enable_cosmos_analytics:
        analytics_parameters = access_parameters(context, apimClientTool, pii_analytics_contract, pii_analytics_policy,
            key_vault=None)

        analytics_params_file = write_parameters(
            repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
            analytics_parameters, run_dir / "analytics")
        utils.format_output('5️⃣.4.3 Generate Parameter File and Deploy Analytics Contract', {'value_1': preview(analytics_params_file)}, status='info', part='RESULT')
    else:
        run.skip("cosmos-section-48", "Optional analytics/permission grant disabled.")


<a id='5.4.4'></a>
#### 5️⃣.4.4 Retrieve API Key for Analytics Contract


### 5️⃣.4.4 Retrieve API Key for Analytics Contract

Deploy the reviewed artifact for 5️⃣.4.4 Retrieve API Key for Analytics Contract, then inspect its real resource outputs and ownership.

**Input:** `enable_cosmos_analytics`, `analytics_params_file`, `location`, `preexisting_products`, `owned_products`, `pii_analytics_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('5️⃣.4.4 Retrieve API Key for Analytics Contract', inputs={'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'analytics_params_file': globals().get('analytics_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'pii_analytics_contract': globals().get('pii_analytics_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if enable_cosmos_analytics:
        analytics_outputs = deploy_contract(analytics_params_file, location, pii_analytics_contract["name"],
            required_outputs=("products", "subscriptions"))
        analytics_product_id, analytics_subscription_name = contract_identity(
            analytics_outputs, preexisting_products, owned_products)
        analytics_api_key = apimClientTool.get_subscription_key(analytics_subscription_name)
    else:
        run.skip("cosmos-section-50", "Optional analytics/permission grant disabled.")


<a id='5.5'></a>
### 5️⃣.5 Send Analytics Summary to LLM (via PII Analytics Access Contract)

Hand the quantitative summary to a language model **through the dedicated PII
Analytics Access Contract** (`analytics_subscription_name`). Because that contract
has no PII processing fragments, the analytics payload — which contains anonymized
PII samples and entity categories — passes through unmodified and does not re-trigger
detection or double-anonymization.

Set `send_to_llm = False` to skip the network call and just inspect the payload.


### 5️⃣.5 Send Analytics Summary to LLM (via PII Analytics Access Contract)

Query or inspect 5️⃣.5 Send Analytics Summary to LLM (via PII Analytics Access Contract) evidence for this run; configuration and HTTP success alone are not evidence.

**Input:** `enable_cosmos_analytics`, `model_name`, `pii_df`, `pii_analytics_summary`, `chat_completions_url`

**Output:** Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.


In [ ]:
with utils.notebook_step('5️⃣.5 Send Analytics Summary to LLM (via PII Analytics Access Contract)', inputs={'enable_cosmos_analytics': globals().get('enable_cosmos_analytics'), 'model_name': globals().get('model_name'), 'pii_df': globals().get('pii_df'), 'pii_analytics_summary': globals().get('pii_analytics_summary'), 'chat_completions_url': globals().get('chat_completions_url')}, expected='Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.', namespace=globals()):
    if enable_cosmos_analytics:
        # Guard: ensure 5.4 PII Analytics contract was deployed and key resolved
        if 'analytics_api_key' not in globals() or not analytics_api_key:
            raise RuntimeError(
                "PII Analytics contract is not deployed yet. Run section 5.4 first "
                "to provision and resolve the analytics contract API key."
            )

        # Toggle to actually call the LLM
        send_to_llm = True

        # Model to reason about the analytics
        analytics_model_name = model_name

        if pii_df.empty or not pii_analytics_summary:
            run.blocked("cosmos-llm-analysis", "Optional analytics enabled but Cosmos query returned no records.")
        elif not send_to_llm:
            utils.print_info("send_to_llm = False — skipping LLM reasoning step.")
        else:
            # Compact summary payload
            analytics_payload_for_llm = {
                "summary"        : pii_analytics_summary,
                "sample_records" : [
                    {
                        "id"                : r["id"],
                        "timestamp"         : r["timestamp"],
                        "product_id"        : r["product_id"],
                        "subscription_id"   : r["subscription_id"],
                        "entity_count"      : r["entity_count"],
                        "entity_categories" : r["entity_categories"],
                        "anon_performed"    : r["anon_performed"],
                        "deanon_performed"  : r["deanon_performed"],
                        "deanon_success"    : r["deanon_success"],
                        "input_len"         : r["input_len"],
                        "anonymized_len"    : r["anonymized_len"],
                    }
                    for r in pii_df.head(10).to_dict(orient="records")
                ],
            }

            system_prompt = (
                "You are a senior AI Gateway reliability analyst. You will receive a JSON summary of "
                "PII detection / anonymization / deanonymization telemetry from an Azure API Management "
                "based AI gateway. Produce a concise report (markdown) that covers:\n"
                "1. Overall detection effectiveness (entities, categories, coverage).\n"
                "2. Anonymization & deanonymization success/failure patterns.\n"
                "3. Content-size impact of the PII pipeline.\n"
                "4. Notable per-product / per-subscription observations.\n"
                "5. Concrete recommendations to improve detection quality, reduce false positives, "
                "or tune the confidence threshold / entity exclusions.\n"
                "Be quantitative and reference the numbers in the input."
            )

            user_prompt = (
                "Here is the latest PII processing telemetry summary from the gateway. "
                "Please analyze the gateway's PII detection performance.\n\n"
                f"```json\n{json.dumps(analytics_payload_for_llm, indent=2, default=str)}\n```"
            )

            llm_payload = {
                "model": analytics_model_name,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
                "temperature": 0.2,
            }

            utils.print_info(f"Sending PII analytics summary to LLM via PII Analytics contract: {analytics_subscription_name}")
            try:
                start_time = time.time()
                response = utils.http_call(requests.post, 
                    chat_completions_url,
                    headers={"api-key": analytics_api_key},
                    json=llm_payload,
                    timeout=120,
                )
                elapsed_time = time.time() - start_time
                utils.print_response_code(response)

                if response.status_code == 200:
                    data = response.json()
                    llm_analysis = completion_text(data)
                    run.check("cosmos-llm-analysis", bool(llm_analysis), expected="Nonempty analytics completion", actual=response.status_code)
                    utils.print_ok(f"✅ LLM analysis received in {elapsed_time:.2f}s\n")
                    try:
                        from IPython.display import Markdown, display
                        display(Markdown(llm_analysis))
                    except Exception:
                        utils.format_output('5️⃣.5 Send Analytics Summary to LLM (via PII Analytics Access Contract)', {'llm_analysis': llm_analysis}, status='info', part='RESULT')
                else:
                    run.check("cosmos-llm-analysis", False, expected="200 nonempty completion", actual=response.status_code)
                    utils.print_error(f"❌ LLM call failed: {response.text[:500]}")
            except Exception as e:
                run.check("cosmos-llm-analysis", False, expected="200 nonempty completion", actual=str(e))
                utils.print_error(f"❌ Request to LLM failed: {e}")
    else:
        run.skip("cosmos-section-52", "Optional analytics/permission grant disabled.")


---
## 📊 Results Summary
---

### 📊 Results Summary

Collect the final 📊 Results Summary verdict without discarding earlier failed or blocked required cases.

**Input:** `pii_pending`, `pii_state_evidence`, `cosmos_account_endpoint`, `cosmos_container_name`, `evidence_poll_attempts`

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('📊 Results Summary', inputs={'pii_pending': globals().get('pii_pending'), 'pii_state_evidence': globals().get('pii_state_evidence'), 'cosmos_account_endpoint': globals().get('cosmos_account_endpoint'), 'cosmos_container_name': globals().get('cosmos_container_name'), 'evidence_poll_attempts': globals().get('evidence_poll_attempts')}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    evidence_records = list(pii_state_evidence)
    if cosmos_account_endpoint and pii_pending:
        try:
            from azure.cosmos import CosmosClient
            from azure.identity import DefaultAzureCredential
            credential = DefaultAzureCredential() if cosmos_auth_mode == "aad" else cosmos_account_key
            evidence_client = CosmosClient(cosmos_account_endpoint, credential=credential)
            container = evidence_client.get_database_client(cosmos_database_name).get_container_client(cosmos_container_name)
            for attempt in range(evidence_poll_attempts):
                evidence_records = list(container.query_items(
                    query="SELECT * FROM c WHERE c.type = 'PII_Processing' AND c.context.product.id = @product AND CONTAINS(c.context.api.content.originalRequest, @run)",
                    parameters=[{"name": "@product", "value": masking_product_id},
                                {"name": "@run", "value": "validation-" + run_id}],
                    enable_cross_partition_query=True))
                if all(any(item["marker"] in record.get("context", {}).get("api", {}).get("content", {}).get("originalRequest", "")
                           for record in evidence_records) for item in pii_pending):
                    break
                if attempt + 1 < evidence_poll_attempts:
                    time.sleep(evidence_poll_seconds)
        except Exception as exc:
            utils.format_output('📊 Results Summary', {'value_1': "PII evidence query unavailable:", 'value_2': str(exc)}, status='info', part='RESULT')
    for pending in pii_pending:
        verify_pii_state(pending, evidence_records)
    run.finish()


<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Remove the test PII access contracts from APIM created during this notebook session.

> **Note:** This will not delete any created secrets in Azure Key Vault.

### 🧹 Cleanup (Optional)

Apply the explicitly enabled 🧹 Cleanup (Optional) action only to resources owned by this run.

**Input:** `owned_products`, `cleanup_delete_contract`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('🧹 Cleanup (Optional)', inputs={'owned_products': globals().get('owned_products'), 'cleanup_delete_contract': globals().get('cleanup_delete_contract')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    cleanup_products(apimClientTool, owned_products, enabled=cleanup_delete_contract)
